# Latin Hypercube-based Inference Pipeline - Identify fixed SIR model parameters

In this notebook we use synthetic viral read count data from a fully-parameterised toy population to
 - (1) develop the inferential pipeline for which uses the data to inform the non-estimated model parameters (due to identifiability issues) and 
 - (2) assess the quality of the estimiated parameter inference.

The rodent population are assumed to follow the dynamics of the SIR algorithm with a logistic birth term rate with multiple birth pulses. If the estimates of the population model parameters are close to the true model parameter values that characterise the toy population in the first place, this implies the validity of inferential approach, and therefore lend credibility to the results produced when the same pipeline is applied to metaviromic datasets collected through wildlife studies, as done in _James Hay et al. (2021)[1]_.

Similar to field studies, random samples of rodents are drawn from the simulated toy population at predifined sampling times, which satisfy the following:
 - total number of rodents sampled at each time point depends on total population size, using a constant capturing rate, but is capped at a prescribed value;
 - the sampled individuals can be either susceptible (S), infected (I) or recovered (R), with no predefined quantities of each;
 - all individuals sampled are born and alive at the time of sampling.

For each of the sampled individuals, we use the SIR model's embedded `viral_read_model` to produce viral read count data, similar to what data is produced from the field studies (byproduct in our analyses, ground truth in real studies).

For the parameter inference we follow an optimisation approach, using the Bare-bones CMA-ES method from *Pints [2]* for parameter inference. The non-estimated parameters of the model are informed by the available metaviromic data, using a latin hypercube method, that uniformly samples guesses of fixed parameter values from the parameter hyperspace, and then for each guess peformed model fitting to the available metaviromic data; then, using the associated maximum log-likelihood associated with the fitting result, we select those fixed parameter regimes that themselves maximise these values, which gives us a restricted region of possible non-inferred parameter values that the available data informs to be more likely to have produced the data in the first place.

We replicate these analyses for a range of sample sizes and frequencies of sampling values, to compare the quality of parameter estimation and proportion of infected population across different sampling protocols.

**************
### References
[1] James A. Hay et al., _Estimating epidemiologic dynamics from cross-sectional viral load distributions_. Science373,**eabh0635(2021)**. DOI:10.1126/science.abh0635

[2] Clerx, M., Robinson, M., Lambert, B., Lei, C. L., Ghosh, S., Mirams, G. R., & Gavaghan, D. J.,
_Probabilistic Inference on Noisy Time Series (PINTS)_.
Journal of Open Research Software (2019), 7(1), 23. DOI:10.5334/jors.252

In [1]:
# Load necessary libraries
import os
import math
import numpy as np
import pandas as pd
from scipy.stats import multinomial, skew, gumbel_r
import math
import metavirommodel as mm
import metavirommodel.inference as mmi
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pints
from matplotlib import pyplot as plt
import pints.plot

# Choose array of colours for graphs and compartments names
colours = ['blue', 'red', 'green', 'purple', 'orange', 'black', 'gray', 'pink']
compartments = ['S', 'I', 'R']

# Set random seed
np.random.seed(270)

## Gillespie stochastic SIR algorithm with logistic birth term rate

#### Define rodent population

In [2]:
# Set initial reproduction number
R_0 = 3

# Set initial population state S - I - R
N_init = 200
# S_init = int(N_init / R_0)
S_init = 180
I_init = N_init - S_init
R_init = 0
initial_population = [S_init, I_init, R_init]

# Set birth rate
precipitation_data = pd.read_csv(os.path.join('../../data/precipitation/Precipitation.csv'))

theta = mm.BirthRatePrec(precipitation_data, [0.7, 2.8, 30])

# Set death rates
mu = 0.002
nu = 0.003

# Set transition rates
infect_period = 15
beta =  R_0 / infect_period
gamma = 1 / infect_period

# Coalesce into paramater vector
parameters = initial_population
parameters.extend([theta, mu, nu, beta, gamma])

# Instantiate algorithm
algorithm = mm.LogisticGrowthMetaviromodel(carrying_capacity=400)

# Select start and end times
start_time = 1
end_time = 360

times = list(range(start_time, end_time+1))

# Select number of experiments
num_experiments = 1

output_algorithm = []

S_history_algorithm = []
I_history_algorithm = []
R_history_algorithm = []

I_times_history_algorithm = []
R_times_history_algorithm = []

for _ in range(num_experiments):
    output, S_history, I_history, R_history, I_times_history, R_times_history = algorithm.simulate_fixed_times(parameters, start_time, end_time)
    output_algorithm.append(output)

    S_history_algorithm.append(S_history)
    I_history_algorithm.append(I_history)
    R_history_algorithm.append(R_history)

    I_times_history_algorithm.append(I_times_history)
    R_times_history_algorithm.append(R_times_history)

output_algorithm = np.asarray(output_algorithm)

### Plot output of Gillespie for the different compartments

In [3]:
# Trace names - represent the type of individuals for the simulation
trace_name = ['{}'.format(s) for s in compartments]

# Names of panels
panels = ['{} only'.format(s) for s in compartments] + ['Total Population']

fig = go.Figure()
fig = make_subplots(rows=int(np.ceil(len(panels)/2)), cols=2, subplot_titles=tuple('{}'.format(p) for p in panels))

# Add traces to the separate counts panels
for s, spec in enumerate(compartments):
    fig.add_trace(
        go.Scatter(
            y=np.mean(output_algorithm[:, :, s], axis=0).tolist(),
            x=times,
            mode='lines',
            name=trace_name[s],
            line_color=colours[s]
        ),
        row= int(np.floor(s / 2)) + 1,
        col= s % 2 + 1
    )

fig.add_trace(
    go.Scatter(
        y=np.mean(np.sum(output_algorithm, axis=2), axis=0).tolist(),
        x=times,
        mode='lines',
        name='Total Population',
        line_color='black'
    ),
    row= 2,
    col= 2
)

# Add axis labels
fig.update_layout(
    title='Counts of compartments over time:<br>IC = {}, θ = {}, μ = {}, v = {}, β = {:.2f}, γ = {:.2f}'.format(parameters[0:3], parameters[3], parameters[4], parameters[5], parameters[6], parameters[7]),
    width=1100, 
    height=600,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis2=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis2=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis3=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis3=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis4=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis4=dict(
        linecolor='black',
        title = 'Individuals')
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Logistic_VR_SIR-gillespie.pdf')
fig.show()

## Produce Viral read counts values

In [4]:
# Set parameter for the viral read counts model
t_eclipse = 3  # (0 days) Time from infection to initial viral growth
t_peak = 7  # (5 days ) Time from initial viral growth to peak viral load
t_switch = 5  # (9.38 days) Time from peak viral load to secondary waning phase
t_mod = 15  # (14 days) time from secondary waning phase until gumbel distribution reaches its min scale parameter
t_LOD = math.inf # ( inf days ) Time from infection until modal read counts value is equal to the limit of detection

sigma_obs = .25  # Initial scale parameter for the Gumbel distribution until a=teclipse+tpeak+tswitch
s_mod = 0.4  # 0.4 multiplicative factor applied to scale paramter for the Gumble distrbution - starting at t_eclipse + t_peak + t_switch + t_scle
v_zero = 2  # read counts value at time of infection
v_peak = 388  # (20) Modal read counts value at peak viral load
v_switch = 18  # (33) Modal read counts value at a = teclipse + tpeak + tswitch
v_LOD = 2  # Limit of detection of read counts value

parameters_vl = [
    t_eclipse, t_peak, t_switch, t_mod, t_LOD,
    v_zero, v_peak, v_switch, v_LOD,
    s_mod, sigma_obs]

# Set read counts value for the suceptible and recovered individuals
VR_susc = 2

### Plot Viral read Model

In [5]:
time_from_infec = np.arange(1, 50)
vr_val = []

for ti in time_from_infec:
    ti_vr_val = []
    for _ in range(10000):
        ti_vr_val.append(algorithm.viral_read_model(parameters_vl, ti))
    vr_val.append(ti_vr_val)

vr_val = np.asarray(vr_val)

In [6]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=time_from_infec,
        x=np.mean(vr_val, axis=1),
        mode='lines',
        name='Mean Viral read',
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        y=time_from_infec.tolist() + time_from_infec.tolist()[::-1],
        x=np.quantile(vr_val, 0.975, axis=1).tolist() + np.quantile(vr_val, 0.025, axis=1).tolist()[::-1],
        mode='lines',
        fill='toself',
        fillcolor='blue',
        line_color='blue',
        opacity=0.3,
        showlegend=False,
    )
)

# Add axis labels
fig.update_layout(
    width=500, 
    height=500,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Mean Viral read',
        autorange='reversed'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Time since infection'),
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Logistic-Viral_read_model.pdf')
fig.show()

### Compute the history of recovered individuals that fully clear the virus and generation times distribution

#### 0 = 'not cleared'; 1 = 'cleared'

In [7]:
# Daily probability of recovered fully clearing the virus
p_addl = 0.2

R_history_clear_algorithm = []

# Go through each run experiment
for _ in range(num_experiments):
    R_history_clear = []

    # Go through each recorded day
    for t, time in enumerate(times):
        current_clear_status = []

        # If there are any recovered individual
        if len(R_times_history_algorithm[_][t]) > 0:
            # Go through each of them and
            for ind, ind_ID in enumerate(R_history_algorithm[_][t]):
                clear_status = 0

                # If they have previously cleared the virus they signal that
                if ind_ID in R_history_algorithm[_][t-1] and R_history_clear[-1][R_history_algorithm[_][t-1].index(ind_ID)] == 1:
                    clear_status = 1
                # if not, they could do it today, if their time since infection exceeds teclipse + tpeak + tswitch
                elif time > R_times_history_algorithm[_][t][ind] + t_eclipse + t_peak + t_switch:
                    clear_status = 1 - np.random.binomial(1, p = (1-p_addl)**(
                        time - R_times_history_algorithm[_][t][ind] - t_eclipse - t_peak - t_switch))

                current_clear_status.append(clear_status)

        R_history_clear.append(current_clear_status)                

    R_history_clear_algorithm.append(R_history_clear)

In [8]:
# Compute the generation times distribution, which also follows a
# left-skewed Gumbel distribution
def create_generation_times(p_addl, parameters_vl):
    t_eclipse, t_peak, t_switch, t_mod, t_LOD, v_zero, v_peak, v_switch, v_LOD, s_mod, sigma_obs = parameters_vl

    generation_times = []

    for _ in range(70):
        if _ < t_eclipse + t_peak + t_switch:
            generation_times.append(
            1-gumbel_r.cdf(
                np.log(v_LOD),
                algorithm._compute_mode_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_LOD,
                    np.log(v_zero), np.log(v_peak), np.log(v_switch), np.log(v_LOD)),
                algorithm._compute_sigma_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_mod,
                    s_mod, sigma_obs)
            ))
        
        else:
            generation_times.append(
            (1-gumbel_r.cdf(
                np.log(v_LOD),
                algorithm._compute_mode_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_LOD,
                    np.log(v_zero), np.log(v_peak), np.log(v_switch), np.log(v_LOD)),
                algorithm._compute_sigma_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_mod,
                    s_mod, sigma_obs)
            )) * (1-p_addl)**(_ - t_eclipse - t_peak - t_switch))

    return generation_times

generation_times = create_generation_times(p_addl, parameters_vl)

#### Plot generation times

In [9]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=time_from_infec,
        y=generation_times,
        mode='lines',
        name='Generation times',
        showlegend=False,
    )
)

fig.show()

## Parameter inference
In this section we test the quality of parameter inference for an optimisation approach, using the Bare-bones CMA-ES method from *Pints [2]* for single parameter inference, for a range of sample sizes and frequencies of sampling values, to compare the quality of parameter estimation and proportion of infected population across different sampling protocols.

#### Sample individuals with specific frequencies and in specific batch sizes

In [ ]:
freq_samplying_range = [3, 5, 7, 14, 21, 28]
capturing_rate_range = [0.05, 0.1, 0.2]

#### Method to create viral read data and ground truth

In [11]:
def sensitivity_analysis_run(sample_points, sample_sizes):
    vr_values = []
    vr_infec = []

    vr_susc_ids = []
    vr_infec_ids = []
    vr_recov_ids = []

    vr_time_of_recov_infec = []
    vr_time_of_infec = []
    vr_time_since_infec = []

    for _ in range(num_experiments):
        experiment_vr_values = []
        experiment_infec = []

        experiment_susc_ids = []
        experiment_infec_ids = []
        experiment_recov_ids = []

        experiment_time_of_recov_infec = []
        experiment_time_of_infec = []
        experiment_time_since_infec = []
        # At each point in time sample sample_size individuals
        for t, time in enumerate(sample_points):
            # Identify the current infections at the specified timepoint
            current_susceptibles = S_history_algorithm[_][time-1]
            current_infections = I_history_algorithm[_][time-1]
            current_recovered = R_history_algorithm[_][time-1]
            current_infection_times = I_times_history_algorithm[_][time-1]
            current_recov_infection_times = R_times_history_algorithm[_][time-1]
            current_recov_clear_virus_status = R_history_clear_algorithm[_][time-1]
            current_recov_clear_virus_status = R_history_clear_algorithm[_][time-1]

            # Sample without replacement the sample_size individuals and
            # determine their time since infection to produce Ct values
            number_selected_susc, number_selected_infec, number_selected_rec = \
                multinomial.rvs(
                    n=sample_sizes[t],
                    p=output_algorithm[_, time-1, :]/np.sum(output_algorithm[_, time-1, :])) # determine how many of those sampled are S, I and R

            # First add the Ct values for the sampled susceptibele and recovered individuals
            sampled_vr_values = [VR_susc] * number_selected_susc

            selected_individuals_susc_ids = np.random.choice(
                    current_susceptibles,
                    size=number_selected_susc,
                    replace=False).tolist() # determine the ids of those sampled Ss
            
            if len(current_recov_infection_times) > 0:
                # If we have at least one selected recovered
                selected_individuals_indices = np.random.choice(
                    range(len(current_recov_infection_times)),
                    size=number_selected_rec,
                    replace=False).tolist() # determine the indices of those sampled Rs
            
                selected_individuals_rec_ids = [current_recovered[_] for _ in selected_individuals_indices]

                # Determine the time of infection of those sampled Rs
                selected_individuals_recov_infec_times = [current_recov_infection_times[_] for _ in selected_individuals_indices]

                sample_time_since_infec = time - selected_individuals_recov_infec_times # determine how long since infection for selected Rs

                # Determine the clearence of infection of those sampled Rs
                selected_individuals_clear_virus_status = [current_recov_clear_virus_status[_] for _ in selected_individuals_indices]

                # Run viral read model to determine individual viral read counts for each sample
                for i, ti in enumerate(sample_time_since_infec):
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, ti) * selected_individuals_clear_virus_status[i])

            elif number_selected_rec > 0:
                # If initial step when no history of infection is provided
                for i in range(number_selected_rec):
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, time))
                
                sample_time_since_infec = np.zeros(number_selected_rec)
                selected_individuals_rec_ids = [] 
            else:
                sample_time_since_infec = []
                selected_individuals_rec_ids = []

            if len(current_infection_times) > 0:
                # If we have at least one selected infection
                selected_individuals_indices = np.random.choice(
                    range(len(current_infection_times)),
                    size=number_selected_infec,
                    replace=False).tolist() # determine the indices of those sampled Is
                
                # Determine the ids of those sampled Is
                selected_individuals_infec_ids = [current_infections[_] for _ in selected_individuals_indices]

                # Determine the time of infection of those sampled Is
                selected_individuals_infec_times = [current_infection_times[_] for _ in selected_individuals_indices]
            
                sample_time_since_infec = time - selected_individuals_infec_times # determine how long since infection for selected Is

                # Run Ct model to determine individual Ct counts for each sample
                for ti in sample_time_since_infec:
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, ti))
            
            elif number_selected_infec > 0:
                # If initial step when no history of infection is provided
                for i in range(number_selected_infec):
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, time))
                
                selected_individuals_infec_times = np.zeros(number_selected_infec)
                sample_time_since_infec = np.zeros(number_selected_infec)
                selected_individuals_infec_ids = [] 
            else:
                selected_individuals_infec_times = []
                sample_time_since_infec = []
                selected_individuals_infec_ids = [] 

            experiment_vr_values.append(sampled_vr_values)
            experiment_infec.append(number_selected_infec)
            
            experiment_susc_ids.append(selected_individuals_susc_ids)
            experiment_infec_ids.append(selected_individuals_infec_ids)
            experiment_recov_ids.append(selected_individuals_rec_ids)

            experiment_time_of_recov_infec.append(selected_individuals_recov_infec_times)
            experiment_time_of_infec.append(selected_individuals_infec_times)
            experiment_time_since_infec.append(sample_time_since_infec)
        
        vr_values.append(experiment_vr_values)
        vr_infec.append(experiment_infec)

        vr_susc_ids.append(experiment_susc_ids)
        vr_infec_ids.append(experiment_infec_ids)
        vr_recov_ids.append(experiment_recov_ids)

        vr_time_of_recov_infec.append(experiment_time_of_recov_infec)
        vr_time_of_infec.append(experiment_time_of_infec)
        vr_time_since_infec.append(experiment_time_since_infec)

    vr_time_of_infec_data = []

    for _ in range(num_experiments):
        experiment_vr_time_of_infec_data = pd.DataFrame(columns=['ID', 'Value'])
        for t, time in enumerate(sample_points):
            experiment_vr_time_of_infec_data = pd.concat(
                [
                    experiment_vr_time_of_infec_data,
                    pd.DataFrame({
                        'ID': vr_susc_ids[_][t] + vr_recov_ids[_][t] + vr_infec_ids[_][t],
                        'Value': [400] * len(vr_susc_ids[_][t]) + vr_time_of_recov_infec[_][t] + vr_time_of_infec[_][t]
                    })
                ])

        vr_time_of_infec_data.append(experiment_vr_time_of_infec_data)

    vr_values_data = []

    for _ in range(num_experiments):
        experiment_vr_values_data = pd.DataFrame(columns=['ID', 'TimeOfSample', 'Value'])
        for t, time in enumerate(sample_points):
            experiment_vr_values_data = pd.concat(
                [
                    experiment_vr_values_data,
                    pd.DataFrame({
                        'ID': vr_susc_ids[_][t] + vr_recov_ids[_][t] + vr_infec_ids[_][t],
                        'TimeOfSample': [time] * sample_sizes[t],
                        'Value': vr_values[_][t]
                    })
                ])

        vr_values_data.append(experiment_vr_values_data)

    shody_recov_freq = []

    for t in range(len(vr_values[0])):
        shody_recov_freq.append(np.divide(
            (np.where(
                (np.asarray(vr_values[0][t]) > 130) & (np.asarray(vr_values[0][t]) < 150))[0]).shape[0], 
            sample_sizes))

    return vr_values_data, shody_recov_freq, np.divide(vr_infec[0], sample_sizes)

In [12]:
# Transform birth rate and death rates into function format for inference method
parameters[3] = lambda _: theta(_)
parameters[4] = lambda _: mu
parameters[5] = lambda _: nu

#### Method to run inference with viral read data and plot inferred trajectories against ground truth

In [13]:
def LHC_routine_run(freq_samplying, capturing_rate, n_samples=100, n_cores=1):
    sample_points = np.arange(5, 320, freq_samplying)

    # Allow varyiable sample sizes
    sample_sizes = np.array(
        [int(np.floor(min(30, np.sum(output_algorithm[0, time-1, :]) * capturing_rate))) for time in sample_points])
    
    print(sample_sizes)

    # For each choice of sample size and frequency infer parameters: 
    vr_values_data, shody_recov_freq, infec_freq_sample = sensitivity_analysis_run(sample_points, sample_sizes)

    latin_hypercube_algorithm = mmi.MVRHyperParameterSearch(
        algorithm, vr_values_data[0], mmi.LogisticGrowthMVRVirReadInfer,
        [[5], [7], [10], [20],
         [10,5], [3880,10], [90,5],
         [0.5,1], [0.5,2], [2,5]])
    
    best_results, results = latin_hypercube_algorithm.latin_hypercube_search(n_samples=n_samples, n_cores=n_cores)

    # Go through all results and select parameter margins for 95% best results
    for index, row in results.iterrows():
        # Select only rows that are among the top 95% of the results
        likelihood_upper = np.quantile(results['Log-Likelihood'].abs(), 0.25)
        selected_results = results[results['Log-Likelihood'].abs() <= likelihood_upper]

        # Report maximum and minimums for all fixed parameters
        all_max = np.max(selected_results['Fixed Parameter Values'].values.tolist(), axis=0)
        all_min = np.min(selected_results['Fixed Parameter Values'].values.tolist(), axis=0)

        fixed_parameter_boundaries = []
        for _, m in enumerate(all_max):
            fixed_parameter_boundaries.append([all_min[_], all_max[_]])

    print(best_results)
    # Plot best results
    # parameters = Init_cond + [theta, mu, nu, beta, gamma]
    # parameters[4] = best_results['Inferred Parameter Values'].values[0][1]  # mu, if inferred
    parameters[6] = best_results['Inferred Parameter Values'].values[0][0] / infect_period  # beta
    parameters[7] = best_results['Fixed Parameter Values'].values[0][-1]  # gamma

    parameters_vl = best_results['Fixed Parameter Values'].values[0][:-2]
    p_addl = best_results['Fixed Parameter Values'].values[0][-2]
    generation_times = create_generation_times(p_addl, parameters_vl)

    mvr_inference = mmi.LogisticGrowthMVRVirReadInfer(algorithm, generation_times=generation_times)
    mvr_inference.read_viral_read_data(vr_values_data[0], parameters_vl)
    mvr_inference._create_posterior()

    theta_found = []
    infec_found = []
    for _ in range(1000):
        output_found = algorithm.simulate_fixed_times(parameters, start_time, end_time)[0]

        theta_found.append(np.divide(output_found[:, 1], np.sum(output_found, axis=1)))
        infec_found.append(output_found[:, 1])

    theta_found = np.array(theta_found)
    infec_found = np.array(infec_found)

    theta_found_mean = np.mean(theta_found, axis=0)
    theta_found_upper = np.quantile(theta_found, 0.975, axis=0)
    theta_found_lower = np.quantile(theta_found, 0.025, axis=0)

    infec_found_mean = np.mean(infec_found, axis=0)
    infec_found_upper = np.quantile(infec_found, 0.975, axis=0)
    infec_found_lower = np.quantile(infec_found, 0.025, axis=0)

    output_found_det = mvr_inference.loglikelihood._run_sir_model(
        parameters, np.arange(max(mvr_inference.loglikelihood._vr_sampled_times))
    )
    theta_found_det = np.divide(output_found_det[:, 1], np.sum(output_found_det, axis=1))
    infec_found_det = output_found_det[:, 1]

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1] / np.mean(np.sum(output_algorithm, axis=2), axis=0)).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_freq_sample.tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq',
            line_color='blue'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=(1 - np.array(shody_recov_freq)).tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq (extreme I)',
            line_color='green'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=theta_found_upper.tolist() + theta_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Sensitivity analysis: Frequency:{} days; Capturing rate:{}'.format(freq_samplying, capturing_rate),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'),
        )

    fig.write_image('images/LHC_Best_Logistic_Viral_read_CredInt_Freq_{}_Capture_rate_{}.pdf'.format(freq_samplying, capturing_rate))
    fig.show()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1]).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=infec_found_upper.tolist() + infec_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Sensitivity analysis: Frequency:{} days; Capturing rate:{}'.format(freq_samplying, capturing_rate),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'))

    fig.write_image('images/LHC_Best_Total_Infec_Logistic_Viral_read_Sensitivity_Freq_{}_Capture_rate_{}.pdf'.format(freq_samplying, capturing_rate))
    fig.show()

    return fixed_parameter_boundaries

#### Run optimisation-based inference method for multiple sampling protcols

In [14]:
fixed_parameter_boundaries = LHC_routine_run(freq_samplying_range[2], capturing_rate_range[0], 200, 4)

[10 10 10 10 10 10 10 10  9  9  9  9  9 11 14 18 19 19 19 19 19 19 19 19
 19 19 19 19 19 19 18 18 18 19 19 19 19 19 19 19 19 19 19 19 19]


/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning: The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.
  warnings.warn(
/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning: The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.
  warnings.warn(
/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning: The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.
  warnings.warn(
/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning: The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.
  warnings.warn(


Maximising LogPDF
Maximising LogPDF
Using Bare-bones CMA-ES
Using Bare-bones CMA-ESRunning in sequential mode.

Running in sequential mode.
Population size: 4
Population size: 4
Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Iter. Eval. Best      Current   Time    
0     4     -3471.623 -3471.623   0:14.1
Iter. Eval. Best      Current   Time    
0     4     -7864.501 -7864.501   0:14.1
Iter. Eval. Best      Current   Time    
0     4     -3991.426 -3991.426   0:14.2
Iter. Eval. Best      Current   Time    
0     4     -4374.841 -4374.841   0:14.7
1     8     -3470.112 -3470.112   0:28.6
1     8     -7862.587 -7862.587   0:28.7
1     8     -3990.593 -3990.593   0:28.8
1     8     -4374.841 -4390.924   0:29.8
2     12    -3470.112 -3471.937   0:43.5
2     12    -7862.587 -7863.84    0:43.6
2     12    -3990.593 -3991.525   0:43.8
2     12    -4374.841 -4377.2

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/scipy/stats/_continuous_distns.py:4290: RuntimeWarning: overflow encountered in exp
  return np.exp(-np.exp(-x))
/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning: The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.
  warnings.warn(
/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/scipy/stats/_continuous_distns.py:4287: RuntimeWarning: overflow encountered in exp
  return -x - np.exp(-x)


Iter. Eval. Best      Current   Time    
0     4     -37601.95 -37601.95   0:13.5
1     8     -37576.71 -37576.71   0:27.3
2     12    -37576.36 -37576.36   0:41.7
108   432   -4373.356 -4373.356  51:59.1
Halting: No significant change in best function evaluation for 100 iterations.
[3.07395794] -4373.355834056612
Optimisation phase is finished.
                              Fixed Parameter Values  ... Log-Likelihood
0  [6.0, 5.0, 10.0, 29.0, inf, 1.0967595305491205...  ...   -4373.355834

[1 rows x 3 columns]
Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4


...

Iter. Eval. Best      Current   Time    
0     4     -1381.288 -1381.288   0:09.2
60    244   -1702.365 -1702.374   9:42.5
1     8     -1378.552 -1378.552   0:18.3
2     12    -1377.275 -1377.275   0:27.7
3     16    -1376.623 -1376.623   0:37.1
60    244   -1724.874 -1724.875   9:49.9
20    84    -1375.395 -1375.764   3:14.8
80    324   -1702.365 -1702.374  12:49.4
80    324   -1724.874 -1724.874  13:03.6
40    164   -1375.395 -1375.555   6:20.4
100   404   -1702.365 -1702.374  15:56.5
111   444   -1702.365 -1702.374  17:30.9
Halting: No significant change in best function evaluation for 100 iterations.
[2.42534585] -1702.3651771064262
Optimisation phase is finished.
                              Fixed Parameter Values  ... Log-Likelihood
0  [1.0, 1.0, 13.0, 19.0, inf, 2.567839513833747,...  ...   -1702.365177

[1 rows x 3 columns]
Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4


/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning: The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.
  warnings.warn(


Iter. Eval. Best      Current   Time    
0     4     -3611.616 -3611.616   0:21.0
1     8     -3610.643 -3610.643   0:41.8
2     12    -3602.817 -3602.817   1:02.4
3     16    -3602.817 -3607.603   1:23.2
100   404   -1724.874 -1724.874  16:17.8
60    244   -1375.395 -1375.555   9:27.6
119   476   -1724.874 -1724.874  19:10.4
Halting: No significant change in best function evaluation for 100 iterations.
[2.65345944] -1724.8737549745408
Optimisation phase is finished.
                              Fixed Parameter Values  ... Log-Likelihood
0  [2.0, 5.0, 12.0, 18.0, inf, 2.610239679165081,...  ...   -1724.873755

[1 rows x 3 columns]
80    324   -1375.395 -1375.555  12:29.9
20    84    -3599.91  -3604.208   6:38.3
100   404   -1375.395 -1375.555  14:57.9
107   428   -1375.395 -1375.555  15:42.2
Halting: No significant change in best function evaluation for 100 iterations.
[2.97535776] -1375.395471075451
Optimisation phase is finished.
                              Fixed Parameter Values 

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/scipy/stats/_continuous_distns.py:4290: RuntimeWarning: overflow encountered in exp
  return np.exp(-np.exp(-x))
/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning: The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.
  warnings.warn(
/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/scipy/stats/_continuous_distns.py:4287: RuntimeWarning: overflow encountered in exp
  return -x - np.exp(-x)


Iter. Eval. Best      Current   Time    
0     4     -42152.16 -42152.16   0:07.7
1     8     -42127.94 -42127.94   0:15.9
2     12    -42127.94 -42129.75   0:23.4
3     16    -42127.94 -42139.24   0:30.9
20    84    -42124.38 -42125.09   2:43.0
40    164   -3599.91  -3600.904  12:12.8
40    164   -42124.24 -42124.43   5:19.4
60    244   -42124.24 -42124.41   8:12.8
60    244   -3599.91  -3600.891  18:11.3
80    324   -42124.24 -42124.41  10:45.1
100   404   -42124.24 -42124.41  13:19.8
115   460   -42124.24 -42124.41  15:05.4
Halting: No significant change in best function evaluation for 100 iterations.
[2.55248967] -42124.243468357614
Optimisation phase is finished.
                              Fixed Parameter Values  ... Log-Likelihood
0  [5.0, 6.0, 6.0, 26.0, inf, 2.7515759995965405,...  ...  -42124.243468

[1 rows x 3 columns]
80    324   -3599.91  -3600.888  23:30.1
100   404   -3599.91  -3600.888  27:42.7
116   464   -3599.91  -3600.888  30:54.3
Halting: No significant change i

In [14]:
LHC_routine_run(freq_samplying_range[2], capturing_rate_range[2], 200, 4)

[30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30
 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30]


/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_65559/4058485854.py:141: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_65559/4058485854.py:157: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning: The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.
  warnings.warn(
/

Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Iter. Eval. Best      Current   Time    
0     4     -inf      -inf        0:22.7
1     8     -inf      -inf        0:44.5
Iter. Eval. Best      Current   Time    
0     4     -11480.21 -11480.21   0:45.3
Iter. Eval. Best      Current   Time    
0     4     -6277.598 -6277.598   0:45.7
Iter. Eval. Best      Current   Time    
0     4     -10574.08 -10574.08   0:45.8
2     12    -inf      -inf        1:06.7
3     16    -inf      -inf        1:29.0
1     8     -11480.21 -11494.7    1:30.7
1     8     -6277.598 -6379.531   1:31.3
1     8     -10574.08 -10577.98   1:31.8
2     12    -11475.38 -11475.38   2:15.6
2     12    -6217.166 -6217.1

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/scipy/stats/_continuous_distns.py:4290: RuntimeWarning: overflow encountered in exp
  return np.exp(-np.exp(-x))
/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning: The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.
  warnings.warn(
/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/scipy/stats/_continuous_distns.py:4287: RuntimeWarning: overflow encountered in exp
  return -x - np.exp(-x)


Iter. Eval. Best      Current   Time    
0     4     -2572.357 -2572.357   0:22.9
1     8     -2453.336 -2453.336   0:46.0
2     12    -2426.861 -2426.861   1:16.3
3     16    -2426.861 -2480.394   1:47.4
20    84    -2423.327 -2423.327  10:15.9
60    244   -11474.16 -11475.09  61:40.7
60    244   -10571.35 -10573.6   62:10.5
60    244   -6198.215 -6198.217  62:12.8
40    164   -2423.314 -2424.069  19:57.4
60    244   -2423.314 -2424.067  29:15.3
80    324   -11474.16 -11475.09  81:02.4
80    324   -6198.215 -6198.215  81:37.3
80    324   -10571.35 -10573.01  81:43.8
80    324   -2423.314 -2424.067  38:15.8
100   404   -2423.314 -2424.067  46:58.3
100   404   -11474.16 -11475.09  99:12.3
100   404   -6198.215 -6198.215  99:49.9
100   404   -10571.35 -10573.01 100:01.7
103   412   -10571.35 -10573.01 101:48.3
Halting: No significant change in best function evaluation for 100 iterations.
[2.79758756] -10571.345369662327
Optimisation phase is finished.
                              Fixed 

...

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning: The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.
  warnings.warn(


Iter. Eval. Best      Current   Time    
0     4     -3107.431 -3107.431   0:11.7
1     8     -3107.431 -3136.754   0:23.3
2     12    -3107.431 -3150.249   0:34.9
3     16    -3106.689 -3106.689   0:46.7
20    84    -3103.668 -3104.084   4:06.8
40    164   -3103.584 -3103.863   8:02.3
60    244   -3103.584 -3103.861  11:53.7
80    324   -3103.584 -3103.861  15:47.5
100   404   -3103.584 -3103.861  19:40.9
113   452   -3103.584 -3103.861  22:01.1
Halting: No significant change in best function evaluation for 100 iterations.
[2.40484425] -3103.584409913931
Optimisation phase is finished.
                              Fixed Parameter Values  ... Log-Likelihood
0  [5.0, 4.0, 9.0, 19.0, inf, 3.9420149506421147,...  ...    -3103.58441

[1 rows x 3 columns]
Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4


/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning: The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.
  warnings.warn(


Iter. Eval. Best      Current   Time    
0     4     -6167.168 -6167.168   0:25.3
1     8     -6167.168 -6226.358   0:50.3
2     12    -6167.168 -6227.563   1:14.8
3     16    -6158.764 -6158.764   1:38.6
20    84    -6158.279 -6162.516   8:33.9
40    164   -6158.248 -6159.286  16:38.1
60    244   -6158.248 -6158.859  24:41.5
80    324   -6158.248 -6158.851  32:40.9
100   404   -6158.248 -6158.851  40:39.7
104   416   -6158.248 -6158.851  41:51.5
Halting: No significant change in best function evaluation for 100 iterations.
[3.27799305] -6158.247540449613
Optimisation phase is finished.
                              Fixed Parameter Values  ... Log-Likelihood
0  [7.0, 11.0, 9.0, 16.0, inf, 1.1900092414744134...  ...    -6158.24754

[1 rows x 3 columns]
[                              Fixed Parameter Values  \
0  [3.0, 7.0, 6.0, 17.0, inf, 1.2522047435150325,...   
0  [2.0, 9.0, 5.0, 16.0, inf, 1.0140753813027177,...   
0  [5.0, 8.0, 6.0, 18.0, inf, 2.3893270979967505,...   
0  [6.0, 8.0,

[[np.float64(1.0), np.float64(13.0)],
 [np.float64(2.0), np.float64(14.0)],
 [np.float64(6.0), np.float64(19.0)],
 [np.float64(12.0), np.float64(33.0)],
 [np.float64(inf), np.float64(inf)],
 [np.float64(2.0010114121051994), np.float64(4.010575370130481)],
 [np.float64(371.93577359955395), np.float64(401.4313676458649)],
 [np.float64(12.37617713340994), np.float64(20.78739831238927)],
 [np.float64(2.0010114121051994), np.float64(4.010575370130481)],
 [np.float64(3.783054358794377e-05), np.float64(2.09402918193643)],
 [np.float64(0.033724355470406656), np.float64(1.2125013728845615)],
 [np.float64(0.20766382003594605), np.float64(0.9999999985496278)],
 [np.float64(0.029411764705882353), np.float64(0.058823529411764705)]]

In [ ]:
LHC_routine_run(freq_samplying_range[5], capturing_rate_range[0], 200, 4)

In [ ]:
LHC_routine_run(freq_samplying_range[5], capturing_rate_range[2], 200, 4)